# Agentic RAG - Assistant Tourisme Maroc

Système RAG agentique construit avec LangGraph pour répondre à des questions sur le tourisme au Maroc.

**Domaine :** Tourisme au Maroc  
**LLM :** Groq (llama-3.3-70b-versatile)  
**Embeddings :** HuggingFace (all-MiniLM-L6-v2)  
**Vector Store :** FAISS  
**Agent Framework :** LangGraph

## 1. Imports et Configuration

In [ ]:
import os
import time
from dotenv import load_dotenv

# LangChain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_community.vectorstores import FAISS
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage
from langchain_core.tools import tool

# LangGraph
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode

# Typing
from typing import Annotated, TypedDict, List

load_dotenv(override=True)
print("Imports OK")

## 2. Construction de la Base Documentaire

Chargement des documents texte sur le tourisme au Maroc, découpage en chunks et vectorisation avec FAISS.

In [ ]:
# Chargement des documents
loader = DirectoryLoader(
    "documents/",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
documents = loader.load()
print(f"Documents chargés : {len(documents)}")
for doc in documents:
    print(f"  - {doc.metadata['source']}")

In [ ]:
# Découpage en chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50
)
chunks = splitter.split_documents(documents)
print(f"Nombre de chunks : {len(chunks)}")
print(f"\nExemple de chunk :")
print(chunks[0].page_content[:200])

In [ ]:
# Vectorisation avec HuggingFace embeddings et stockage FAISS
print("Chargement du modèle d'embeddings...")
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

print("Création de l'index FAISS...")
vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Base vectorielle créée avec succès !")

## 3. Développement des Outils (Tools)

Création de l'outil de recherche documentaire que l'agent pourra utiliser.

In [ ]:
@tool
def recherche_tourisme(query: str) -> str:
    """Recherche des informations touristiques sur le Maroc dans la base documentaire.
    Utilise cette fonction pour répondre aux questions sur les villes, attractions,
    hébergements, transports et gastronomie du Maroc."""
    docs = retriever.invoke(query)
    if not docs:
        return "Aucune information trouvée pour cette requête."
    
    result = ""
    for i, doc in enumerate(docs):
        source = doc.metadata.get('source', 'inconnu').split('\\')[-1].replace('.txt', '')
        result += f"[Source: {source}]\n{doc.page_content}\n\n"
    return result

tools = [recherche_tourisme]
print("Outil 'recherche_tourisme' créé")

# Test de l'outil
print("\nTest de l'outil :")
print(recherche_tourisme.invoke("Que visiter à Marrakech ?")[:300])

## 4. Architecture du Graphe LangGraph

Construction du graphe agentique avec état, mémoire et logique de décision.

In [ ]:
# Définition du State (état partagé entre les noeuds)
class AgentState(TypedDict):
    messages: Annotated[list, add_messages]

# Initialisation du LLM avec les outils
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)
llm_with_tools = llm.bind_tools(tools)

# Noeud Agent : décide si utiliser un outil ou répondre directement
def agent_node(state: AgentState):
    system = SystemMessage(content="""Tu es un assistant expert en tourisme au Maroc.
Tu aides les voyageurs à planifier leur séjour en fournissant des informations précises
sur les villes, attractions, hébergements, transports et gastronomie.
Utilise l'outil de recherche pour trouver des informations pertinentes avant de répondre.
Réponds toujours en français de manière claire et structurée.""")
    
    messages = [system] + state["messages"]
    response = llm_with_tools.invoke(messages)
    return {"messages": [response]}

# Fonction de décision : continuer vers les outils ou terminer
def should_continue(state: AgentState):
    last_message = state["messages"][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    return END

print("Noeuds définis")

In [ ]:
# Construction du graphe
workflow = StateGraph(AgentState)

# Ajout des noeuds
workflow.add_node("agent", agent_node)
workflow.add_node("tools", ToolNode(tools))

# Point d'entrée
workflow.set_entry_point("agent")

# Edges (connexions)
workflow.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END}
)
workflow.add_edge("tools", "agent")

# Compilation du graphe
app = workflow.compile()
print("Graphe compilé avec succès !")

## 5. Visualisation du Graphe

In [ ]:
from IPython.display import Image, display

try:
    graph_image = app.get_graph().draw_mermaid_png()
    display(Image(graph_image))
except Exception as e:
    # Affichage du graphe en format texte si l'image échoue
    print("Structure du graphe :")
    print(app.get_graph().draw_mermaid())

## 6. Fonction de Chat

Fonction principale pour interagir avec l'agent.

In [ ]:
def ask(question: str) -> dict:
    """Pose une question à l'agent et retourne la réponse avec métriques."""
    start = time.time()
    
    result = app.invoke({
        "messages": [HumanMessage(content=question)]
    })
    
    duration = round(time.time() - start, 2)
    answer = result["messages"][-1].content
    
    return {
        "question": question,
        "answer": answer,
        "time": duration
    }

# Test rapide
print("Agent prêt. Test :")
r = ask("Quelle est la meilleure ville à visiter au Maroc ?")
print(f"Réponse ({r['time']}s) : {r['answer'][:200]}...")

## 7. Évaluation du Système

### 7.1 Questions Simples (10 questions)

In [ ]:
questions_simples = [
    "Pourquoi Marrakech est-elle appelée la ville rouge ?",
    "Quelles sont les tanneries les plus anciennes du monde ?",
    "Pourquoi les maisons de Chefchaouen sont-elles peintes en bleu ?",
    "Quelle est la hauteur du minaret de la mosquée Hassan II ?",
    "Quelle est la plus ancienne université du monde encore en activité ?",
    "Quelle ville marocaine est connue pour ses plages et son surf ?",
    "Quel est le plat typique de Fès ?",
    "À quelle distance se trouve l'aéroport Mohammed V de Casablanca ?",
    "Quelle catastrophe a frappé Agadir en 1960 ?",
    "Combien de kilomètres fait la plage d'Agadir ?"
]

resultats_simples = []
print("=== QUESTIONS SIMPLES ===")
for i, q in enumerate(questions_simples, 1):
    print(f"\nQ{i}: {q}")
    r = ask(q)
    resultats_simples.append(r)
    print(f"Réponse ({r['time']}s): {r['answer'][:300]}")
    print("-" * 60)

### 7.2 Questions Complexes (10 questions)

In [ ]:
questions_complexes = [
    "Je veux visiter Marrakech et Fès en une semaine avec un budget limité. Quels hébergements me conseilles-tu et comment me déplacer entre les deux villes ?",
    "Compare Marrakech et Chefchaouen pour un voyage photographique. Laquelle recommandes-tu et pourquoi ?",
    "Quelle est la meilleure période pour visiter le Maroc si je veux éviter la chaleur et les foules touristiques ?",
    "Je suis passionné de surf et de culture. Quelle combinaison de villes me conseilles-tu pour un séjour de 10 jours ?",
    "Explique les différences architecturales entre la médina de Fès et celle de Marrakech, et leur importance historique.",
    "Quelles excursions peut-on faire depuis Agadir pour découvrir la culture berbère ?",
    "Comment planifier un circuit de 2 semaines pour visiter les 5 villes (Casablanca, Marrakech, Fès, Chefchaouen, Agadir) de manière optimale ?",
    "Quels sont les plats traditionnels incontournables à goûter dans chaque ville marocaine ?",
    "Pour un voyage en famille avec des enfants, quelle ville marocaine offre le meilleur rapport activités-sécurité-hébergement ?",
    "Analyse les avantages et inconvénients de visiter Casablanca par rapport aux autres villes impériales du Maroc."
]

resultats_complexes = []
print("=== QUESTIONS COMPLEXES ===")
for i, q in enumerate(questions_complexes, 1):
    print(f"\nQ{i}: {q}")
    r = ask(q)
    resultats_complexes.append(r)
    print(f"Réponse ({r['time']}s): {r['answer'][:400]}")
    print("-" * 60)

## 8. Analyse des Résultats

In [ ]:
import statistics

temps_simples = [r['time'] for r in resultats_simples]
temps_complexes = [r['time'] for r in resultats_complexes]

print("=" * 50)
print("ANALYSE DES PERFORMANCES")
print("=" * 50)

print(f"\nQuestions simples ({len(resultats_simples)} questions):")
print(f"  Temps moyen    : {statistics.mean(temps_simples):.2f}s")
print(f"  Temps minimum  : {min(temps_simples):.2f}s")
print(f"  Temps maximum  : {max(temps_simples):.2f}s")

print(f"\nQuestions complexes ({len(resultats_complexes)} questions):")
print(f"  Temps moyen    : {statistics.mean(temps_complexes):.2f}s")
print(f"  Temps minimum  : {min(temps_complexes):.2f}s")
print(f"  Temps maximum  : {max(temps_complexes):.2f}s")

print(f"\nTotal questions testées : {len(resultats_simples) + len(resultats_complexes)}")
print(f"Temps total d'évaluation : {sum(temps_simples + temps_complexes):.2f}s")